In [ ]:
!pip install -q -U transformers

!pip install -q \
  accelerate \
  bitsandbytes \
  peft \
  datasets \
  torchvision \
  pillow \
  bert-score \
  rouge-score

import os
os.kill(os.getpid(), 9)  # Restart the kernel cleanly

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 24.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account 

In [2]:
import os
import torch
from transformers import (
    ViltProcessor, ViltForQuestionAnswering,
    BitsAndBytesConfig, Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from torch.utils.data import DataLoader
from transformers import DefaultDataCollator

2025-05-18 17:05:05.493645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747587905.689642     106 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747587905.747919     106 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
train_dataset = load_dataset(
    "csv",
    data_files = "/kaggle/input/vqa-dataset/Subset/combined_vqa_single_answer.csv",
    split = "train"
)
test_dataset = load_dataset(
    "csv",
    data_files = "/kaggle/input/vqa-inference-dataset/Inference/combined_inference_vqa_single_answer.csv",
    split = "train"
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
# 1) Overwrite image_path in-place via map()
train_dataset = train_dataset.map(
    lambda example: {
        "image_path": f"/kaggle/input/vqa-dataset/Subset/images/{example['image_path']}"
    }
)

test_dataset = test_dataset.map(
    lambda example: {
        "image_path": f"/kaggle/input/vqa-inference-dataset/Subset/images/{example['image_path']}"
    }
)

Map:   0%|          | 0/8441 [00:00<?, ? examples/s]

Map:   0%|          | 0/2969 [00:00<?, ? examples/s]

In [7]:
# Load processor
processor  = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")

# BitsAndBytesConfig for 8-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",           # best quality/size tradeoff
    bnb_4bit_compute_dtype = torch.float16
)

model = ViltForQuestionAnswering.from_pretrained(
    "dandelin/vilt-b32-finetuned-vqa",
    quantization_config = bnb_config,
    device_map = "auto",                   # sprinkle layers across GPU/CPU
    torch_dtype = torch.float16,           # ensure weights are stored in fp16
    offload_folder = "/kaggle/working/offload",  # CPU offload cache
    offload_state_dict = True
)

# Disable key-value caching (saves VRAM)
model.config.use_cache = False

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Prepare for int8 training
model = prepare_model_for_kbit_training(model)

preprocessor_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/136k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/470M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

In [8]:
# LoRA configuration
lora_config = LoraConfig(
    task_type = "QUESTION_ANS",
    inference_mode = False,
    r = 8,
    lora_alpha = 32,
    target_modules = ["query", "value"],
    lora_dropout = 0.1
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 294,912 || all params: 117,883,449 || trainable%: 0.2502


In [9]:
from PIL import Image

def preprocess_function(example):
    image = Image.open(example['image_path']).convert("RGB")
    encoding = processor(
        image,
        example["question"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    )

    # Use the text label as target (you can tokenize it if your model needs token labels)
    encoding["labels"] = processor.tokenizer(
        example["answer"],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )["input_ids"]

    # Remove batch dimension
    encoding = {k: v.squeeze(0) for k, v in encoding.items()}
    return encoding

In [10]:
train_ds = train_dataset.with_transform(preprocess_function)
test_ds  = test_dataset.with_transform(preprocess_function)

# DataCollator that just pads and stacks what your processor produces
data_collator = DefaultDataCollator()

In [11]:
# Enable gradient checkpointing to reduce memory
model.gradient_checkpointing_enable()

#label_column = "labels"  # or whatever your data preprocessing outputs

training_args = TrainingArguments(
    output_dir='/kaggle/working/results',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=1,
    fp16=True,
    load_best_model_at_end=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=processor,  # works if using processor
)

trainer.train()

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
!!nvidia-smi -l 5

In [ ]:
model.save_pretrained("/kaggle/working/vilt_vqa_quant_lora")

tokenizer.save_pretrained("/kaggle/working/vilt_vqa_quant_lora")